In [1]:

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
import pycircstat2 as cs2
ss = hf.settings_dict()

In [2]:
baseline_tmin = -0.5
baseline_tmax = 0.0
stimulus_tmin = 0.7
stimulus_tmax = 3.7



In [3]:
from scipy.stats import ranksums
from pycircstat2.hypothesis import wheeler_watson_test, watson_u2_test

for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-event-{event_name}-hilbert-vox-ref-vol.stc-stc.h5"
        stc = mne.read_source_estimate(hilbert_stc_file)

        stc_baseline = stc.copy().crop(baseline_tmin, baseline_tmax)
        stc_stimulus = stc.copy().crop(stimulus_tmin, stimulus_tmax)


        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        plv_map = np.zeros(n_voxels)

        for v in range(n_voxels):

            # Extract complex analytic signal
            stim = stc_stimulus.data[v]    # shape: (n_times, n_trials)
            base = stc_baseline.data[v]

            # Compute imaginary cPLV (correct way)
            plv_stim = np.abs(np.mean(np.imag(stim)))   # mean across trials
            plv_base = np.abs(np.mean(np.imag(base)))

            # Difference
            plv_map[v] = plv_stim - plv_base


        print(f"delta_r min  : {plv_map.min():.4f}")
        print(f"delta_r max  : {plv_map.max():.4f}")
        print(f"delta_r mean : {plv_map.mean():.4f}")
        print(f"Voxels > 0   : {(plv_map > 0).sum()} / {n_voxels}")

        z_stc       = stc.copy().crop(0, 0.0 + 1/ss['fs'])
        z_stc.data  = plv_map[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)


loading dataset for subject:  0005_3SJ
delta_r min  : -0.8872
delta_r max  : 0.7342
delta_r mean : -0.0628
Voxels > 0   : 575 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.7941
delta_r max  : 0.6279
delta_r mean : -0.1060
Voxels > 0   : 443 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.8674
delta_r max  : 0.7994
delta_r mean : -0.0932
Voxels > 0   : 570 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.7496
delta_r max  : 0.7842
delta_r mean : 0.0236
Voxels > 0   : 738 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.8516
delta_r max  : 0.8770
delta_r mean : -0.0311
Voxels > 0   : 617 / 1440
